# Introduction to Apache Spark: Philosophy, Ecosystem & Primitives

**Module**: 01_spark_fundamentals  
**Notebook**: 01_introduction_to_spark.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: The foundational concepts of Apache Spark, why it replaced Hadoop MapReduce, and its unified computational engine.
- **Why does this topic exist?**: Understanding Spark's design philosophy clarifies why DataFrames are faster than Python scripts and how distributed execution works.
- **What problem does it solve?**: Traditional tools (Pandas, raw Python, MapReduce) fail to scale beyond single-machine RAM or incur massive disk serialization bottlenecks.
- **Where is it used in real-world Data Engineering?**: Used at Netflix, Uber, Apple, and banks for petabyte-scale ETL, fraud detection, and recommendation pipelines.

---

## 2. The Evolution of Big Data Processing: MapReduce vs Spark

```
Hadoop MapReduce (Disk-Bound Iteration):
[Input] -> [Map] -> [Disk Spill] -> [Network Shuffle] -> [Reduce] -> [Disk Write]
            |                                                           |
            +----------------- Repeat for Next Job <--------------------+
            (Massive Disk I/O Bottleneck at Every Iteration)

Apache Spark (In-Memory DAG Pipeline):
[Input] -> [Transform 1] -> [Transform 2] -> [In-Memory Cache] -> [Action Output]
            \__________________________/
              Fused into single Stage
               (10x - 100x Faster)
```

Spark's core innovations:
1. **In-Memory Computing**: Retains intermediate state in RAM across operations.
2. **DAG (Directed Acyclic Graph)**: Plans multi-step pipelines holistically before running.
3. **Unified Engine**: One unified API for SQL, streaming, machine learning, and graphs.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Spark_Introduction_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Hands-On: Classic Distributed Word Count

Let us run the classic distributed word count on `/opt/spark-data/words.txt`.

In [ ]:
# Read text file as RDD
lines_rdd = spark.sparkContext.textFile("/opt/spark-data/words.txt")
print(f"Total lines in words.txt: {lines_rdd.count()}")

# Tokenize, map, and reduce
word_counts = (
    lines_rdd
    .flatMap(lambda line: line.lower().split())
    .map(lambda word: (word.strip(".,!?\"'"), 1))
    .filter(lambda pair: len(pair[0]) > 0)
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda pair: pair[1], ascending=False)
)

print("Top 10 most frequent words:")
for word, count in word_counts.take(10):
    print(f"  {word:15}: {count}")

## 4. The Same Problem in Spark SQL / DataFrame API

Now let us execute the exact same analysis using the DataFrame API and compare readability and optimization.

In [ ]:
from pyspark.sql import functions as F

words_df = spark.read.text("/opt/spark-data/words.txt")

df_counts = (
    words_df
    .select(F.explode(F.split(F.lower(F.col("value")), "\\s+")).alias("raw_word"))
    .select(F.regexp_replace(F.col("raw_word"), "[^a-zA-Z]", "").alias("word"))
    .filter(F.length(F.col("word")) > 0)
    .groupBy("word")
    .count()
    .orderBy(F.desc("count"))
)

print("DataFrame Word Count Output:")
df_counts.show(10)

---

## Experiment: RDD vs DataFrame Execution Plan Comparison

- **1. What are we testing?**: Compare how Spark optimizes DataFrame operations vs low-level RDD transformations.
- **2. What do we expect?**: DataFrame plan shows Catalyst optimization (WholeStageCodegen, HashAggregate), while RDD plan is opaque.

In [ ]:
print("=== DataFrame Physical Plan ===")
df_counts.explain()

print("\n=== RDD Lineage Debug String ===")
print(word_counts.toDebugString().decode('utf-8'))

**3. What happened?**: DataFrame execution plan shows optimized operators (*HashAggregate, *Project) whereas RDD shows raw stage steps.  
**4. Why did it happen?**: Catalyst optimizes DataFrame operations because it understands relations and column semantics.  
**5. Practical Engineering Lesson**: DataFrames allow Catalyst and Tungsten to optimize execution, whereas RDDs treat user functions as opaque black boxes.

---

## Practical Exercises

### Exercise 1 (Beginner): Filter High-Frequency Words
**Problem Statement**: Filter df_counts to keep only words appearing 2 or more times and print the count of distinct words.  
**Input / Context**: df_counts DataFrame from earlier.

In [ ]:
# Exercise 1: Filter words with count >= 2

### Exercise 2 (Intermediate): Calculate Average Word Length
**Problem Statement**: Using DataFrame API, calculate the average length of all distinct words in words.txt.  
**Input / Context**: df_counts DataFrame

In [ ]:
# Exercise 2: Average word length

### Exercise 3 (Advanced): Monte Carlo Pi Estimation with Parallelize
**Problem Statement**: Estimate Pi using 1,000,000 random points distributed across RDD partitions.  
**Input / Context**: Random sampling with math.hypot(x, y) <= 1.0

In [ ]:
# Exercise 3: Distributed Pi calculation

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Filter High-Frequency Words):
```python
frequent_words = df_counts.filter(F.col("count") >= 2)
print("Count of frequent words:", frequent_words.count())
frequent_words.show()
```

#### Solution 2 (Calculate Average Word Length):
```python
avg_len = df_counts.select(F.avg(F.length(F.col("word"))).alias("avg_length")).collect()[0][0]
print(f"Average distinct word length: {avg_len:.2f}")
```

#### Solution 3 (Monte Carlo Pi Estimation with Parallelize):
```python
import random

def sample_point(_):
    x, y = random.random(), random.random()
    return 1 if (x*x + y*y) <= 1.0 else 0

SAMPLES = 500000
count_inside = spark.sparkContext.parallelize(range(SAMPLES), 8).map(sample_point).reduce(lambda a, b: a + b)
pi_estimate = 4.0 * count_inside / SAMPLES
print(f"Estimated Pi: {pi_estimate:.5f} (True Pi: 3.14159)")
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Spark Paradigm**: Spark eliminates disk bottlenecks by pipelining transformations in memory using DAG scheduling.
- **Unified API**: DataFrames provide a single high-level abstraction optimized by Catalyst across batch, streaming, and SQL.
- **RDD vs DataFrame**: Prefer DataFrames in production; use RDDs only for custom unstructured data manipulation.

### Essential Technical Terms
- `RDD`: Resilient Distributed Dataset (immutable, distributed collection of objects).
- `DAG`: Directed Acyclic Graph representing the dependency graph of transformations.
- `Action`: Terminal operation triggering execution across the cluster.

### Common Engineering Mistakes
- **Using RDDs for Tabular Data**: Misses Catalyst predicate pushdown, column pruning, and Tungsten code generation.
- **Imperative Python Loops**: Iterating over DataFrame rows with Python `for` loops bypasses distributed execution.

### Interview Questions on This Topic
- **Q: Why is Spark faster than MapReduce?**
  - *A*: Spark retains intermediate data in memory across stages and fuses narrow transformations into single tasks using DAG pipelining.
- **Q: What is an RDD?**
  - *A*: An immutable, lazily evaluated, partitioned collection of records that can be recomputed upon node failure via its lineage.

### What to Study Next
Open 01_spark_fundamentals/02_spark_architecture.ipynb to dissect the Driver, Master, Workers, and Executors.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')